In [2]:
%pip install pefile pandas

  Using cached pefile-2024.8.26-py3-none-any.whl (74 kB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import math
import pefile
import pandas as pd

def calc_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = [data.count(bytes([i])) for i in range(256)]
    probs  = [c/len(data) for c in counts if c]
    return -sum(p * math.log2(p) for p in probs)

# ─── Config ─────────────────────────────────────────────────────────────────────
MANIFEST_CSV = r"C:\samples\manifest.csv"
OUTPUT_CSV   = r"C:\samples\static_feature_matrix_updated.csv"
# List of specific APIs and DLLs to flag
APIS = [
    "WriteFile","CreateProcessW","CreateDirectory","sleep",
    "GetProcAddress","CreatePopupMenu","CloseClipboard",
    "LoadLibraryA","GetModuleHandleA"
]
DLLS = [
    "msvcpl40.dll","user32.dll","vcruntime140.dll",
    "oleaut32.dll","kernel.dll","mscore.dll","advapi.dll"
]
# Standard section names
STD_SECTIONS = {".text", ".data", ".rdata", ".rsrc", ".idata", ".pdata", ".reloc"}
# ────────────────────────────────────────────────────────────────────────────────

# load manifest
df = pd.read_csv(MANIFEST_CSV)

rows = []
for _, row in df.iterrows():
    path   = row.FullPath
    label  = row.Label
    mclass = row.Malware_Class

    try:
        pe = pefile.PE(path)
    except Exception:
        # skip non‐PE or corrupted files
        continue

    # ── Header Fields ─────────────────────────────────────
    fh = pe.FILE_HEADER
    oh = pe.OPTIONAL_HEADER

    header_feats = {
        "MajorLinkerVersion":      oh.MajorLinkerVersion,
        "MinorLinkerVersion":      oh.MinorLinkerVersion,
        "SizeOfCode":              oh.SizeOfCode,
        "SizeOfInitializedData":   oh.SizeOfInitializedData,
        "SizeOfUninitializedData": oh.SizeOfUninitializedData,
        "AddressOfEntryPoint":     oh.AddressOfEntryPoint,
        "BaseOfCode":              oh.BaseOfCode,
        "ImageBase":               oh.ImageBase,
        "SectionAlignment":        oh.SectionAlignment,
        "SizeOfHeaders":           oh.SizeOfHeaders,
        "Machine":                 fh.Machine,
        "Magic":                   oh.Magic
    }

    # ── Section Features ───────────────────────────────────
    secs       = pe.sections
    num_sec    = len(secs)
    non_std    = 0
    mismatch   = 0
    entropies  = []

    total_vs   = 0
    total_rs   = 0

    for s in secs:
        name = s.Name.decode(errors="ignore").strip("\x00").lower()
        if name not in STD_SECTIONS:
            non_std += 1
        if s.Misc_VirtualSize != s.SizeOfRawData:
            mismatch += 1

        data = s.get_data()
        ent  = calc_entropy(data)
        entropies.append(ent)

        total_vs += s.Misc_VirtualSize
        total_rs += s.SizeOfRawData

    avg_ent = round(sum(entropies)/num_sec, 3) if num_sec else 0.0
    packed  = int(any(e > 7.0 for e in entropies))

    section_feats = {
        "Num_Sections":           num_sec,
        "Non_Standard_Sections":  non_std,
        "Size_Mismatch_Count":    mismatch,
        "Avg_Entropy":            avg_ent,
        "Packed":                 packed,
        "Total_VirtualSize":      total_vs,
        "Total_RawSize":          total_rs
    }

    # ── Import Features ────────────────────────────────────
    dll_list = []
    api_list = []
    if hasattr(pe, "DIRECTORY_ENTRY_IMPORT"):
        for entry in pe.DIRECTORY_ENTRY_IMPORT:
            dll_list.append(entry.dll.decode(errors="ignore").lower())
            for imp in entry.imports:
                if imp.name:
                    api_list.append(imp.name.decode(errors="ignore"))

    uniq_dlls = set(dll_list)
    uniq_apis = set(api_list)

    # count of all imported (optional)
    import_counts = {
        "Num_Imported_DLLs": len(uniq_dlls),
        "Num_Imported_APIs": len(uniq_apis)
    }

    # binary flags for your chosen DLLs/APIs
    dll_flags = { f"has_{d}": int(d in uniq_dlls) for d in DLLS }
    api_flags = { f"has_{a}": int(a in uniq_apis) for a in APIS }

    # ── Assemble feature row ───────────────────────────────
    row_feats = {
        "Sample":        row.Filename,
        "FullPath":      path,
        "Label":         label,
        "Malware_Class": mclass
    }
    row_feats.update(header_feats)
    row_feats.update(section_feats)
    row_feats.update(import_counts)
    row_feats.update(dll_flags)
    row_feats.update(api_flags)

    rows.append(row_feats)

# ── Write out the full feature matrix ────────────────────────────────
out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"[✔] Wrote {len(out_df)} rows to {OUTPUT_CSV}")

[✔] Wrote 1592 rows to C:\samples\static_feature_matrix_updated.csv


In [4]:
print(os.getcwd())

C:\Users\Windows10


In [3]:
import pandas as pd
data = pd.read_csv(r"C:\Users\Windows10\static_feature_matrix.csv")
data.columns




Index(['Sample', 'FullPath', 'Label', 'Malware_Class', 'Num_Sections',
       'Non_Standard_Sections', 'Size_Mismatch_Count', 'Avg_Entropy', 'Packed',
       'Num_Imported_DLLs', 'Num_Imported_APIs', 'DLLs', 'APIs'],
      dtype='object')

In [4]:
data.head(5)

,Sample,FullPath,Label,Malware_Class,Num_Sections,Non_Standard_Sections,Size_Mismatch_Count,Avg_Entropy,Packed,Num_Imported_DLLs,Num_Imported_APIs,DLLs,APIs
0,0468127a19daf4c7bc41015c5640fe1f,C:\samples\malware\All.ElectroRAT\0468127a19da...,malware,All.ElectroRAT,5,0,5,4.367,0,2,68,"KERNEL32.dll,USER32.dll","CloseHandle,CreateFileW,DecodePointer,DeleteCr..."
1,2a3b92f6180367306d750e59c9b6446b,C:\samples\malware\All.ElectroRAT\2a3b92f61803...,malware,All.ElectroRAT,5,0,5,4.911,0,5,118,"ADVAPI32.dll,KERNEL32.dll,SHELL32.dll,USER32.d...","CloseHandle,CompareStringW,ConvertSidToStringS..."
2,b154ac015c0d1d6250032f63c749f9cf,C:\samples\malware\All.ElectroRAT\b154ac015c0d...,malware,All.ElectroRAT,5,0,5,4.697,0,3,77,"GDI32.dll,KERNEL32.dll,USER32.dll","BeginPaint,CloseHandle,CreateWindowExW,DecodeP..."
3,b96bd6bbf0e3f4f98b606a2ab5db4a69,C:\samples\malware\All.ElectroRAT\b96bd6bbf0e3...,malware,All.ElectroRAT,5,0,5,4.922,0,2,84,"KERNEL32.dll,USER32.dll","BeginPaint,CloseHandle,CreateFileW,CreateWindo..."
4,bb8e52face5b076cc890bbfaaf4bb73e,C:\samples\malware\All.ElectroRAT\bb8e52face5b...,malware,All.ElectroRAT,7,4,6,3.782,0,6,182,"advapi32.dll,gdi32.dll,kernel32.dll,oleaut32.d...","BitBlt,CharLowerBuffA,CharNextA,CharToOemA,Cha..."
